In [ ]:
from dotenv import load_dotenv
import os

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "meta-llama/Llama-3.2-3B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

prompt = "Explain recursion like I'm 10 years old."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

print("Generating response...")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1
)

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[-1]:],
    skip_special_tokens=True
)

print("\nResponse:\n")
print(response)

In [ ]:
%pip install torch transformers accelerate

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "write a quick sort algorithm."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0+7a9abb9
Transformers : 5.9.0
Torch        : 2.12.0+cu130
Triton       : 3.7.0


INFO  Kernel: Auto-selection: adding candidate `AwqMarlinLinear`               


INFO  Kernel: selected -> `AwqMarlinLinear`.                                   


Loading weights: 100%|██████████| 1667/1667 [00:04<00:00, 375.93it/s]
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct-AWQ/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct-AWQ/1ed0a6145da0ce550c628e8e8b678f51e695995d/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct-AWQ/1ed0a6145da0ce550c628e8e8b678f51e695995d/generation_config.json "HTTP/1.1 200 OK"


INFO  Marlin fp16: compiling torch.ops JIT extension in `/home/tobi/.cache/gptqmodel/torch_extensions/marlin_fp16/50006cc31723256a`.


INFO  Marlin fp16: torch.ops JIT compilation failed in 0.9s (estimated ~117s, -116s); using fallback path.


ModuleNotFoundError: Marlin torch.ops kernels are not properly installed. Error: Marlin fp16: failed to build torch.ops JIT extension: CUDA_HOME environment variable is not set. Please set it to your CUDA install root.

In [ ]:
%pip install gptqmodel